# APIs REST en Python — De la teoría al caso práctico

**Curso:** Python for Data Science — Universidad del Pacífico
**Caso práctico:** RAWG Video Games Database API

---

## Cómo está organizado este notebook

Este material tiene **dos partes que están conectadas entre sí**:

| Parte | Qué contiene | Para qué sirve |
|---|---|---|
| **Parte 1 — Teoría** | Los conceptos + mini-demos de 2 o 3 líneas | Entender *qué* pasa y *por qué* |
| **Parte 2 — Caso práctico** | La tarea completa de RAWG resuelta | Ver esos conceptos aplicados de punta a punta |

> **La clave:** cada comentario `#` del caso práctico apunta al concepto
> de la teoría que lo explica, con la marca `[Teoría §N]`.
> Si un `#` no te queda claro, sube a esa sección de la Parte 1.

---

## Índice

**PARTE 1 — TEORÍA**
1. ¿Qué es una API?
2. API vs Web Scraping — ¿cuándo usar cada una?
3. Anatomía de una petición: URL base + endpoint + parámetros
4. La API Key — tu contraseña de acceso
5. Métodos HTTP: GET, POST, PUT, DELETE
6. Códigos de estado — ¿salió bien o mal?
7. JSON — diccionarios y listas disfrazados
8. De JSON a DataFrame
9. Cómo obtener tu API Key de RAWG (paso a paso)
10. Escalabilidad, límites y buenas prácticas

**PARTE 2 — CASO PRÁCTICO: Task 2 — RAWG Video Games Database**
- Part A — General Exploration
- Part B — Category Analysis
- Part C — Comparisons
- Part D — Insights & Conclusions

---
# PARTE 1 — TEORÍA
---

## 1. ¿Qué es una API?

**API** = *Application Programming Interface* → **Interfaz de Programación de Aplicaciones**.

### La analogía del mesero

Imagina que vas a un restaurante:

- Tú eres **el usuario** (tu código en Python).
- La cocina es **la base de datos** de la empresa.
- El **mesero** es la **API**.

Tú no entras a la cocina a servirte solo: sería inseguro, caótico y probablemente
romperías algo. Le pides al mesero, él lleva tu pedido, y te trae exactamente
lo que solicitaste, en un plato ordenado.

> **La API es un intermediario o "mensajero"** entre quien pide los datos
> y la base de datos que los guarda. Es la **puerta autorizada** para hacer solicitudes.

### ¿Por qué existen?

| Sin API | Con API |
|---|---|
| Acceso directo a la base de datos | Acceso controlado por endpoints |
| Inseguro (ves todo) | Seguro (ves solo lo permitido) |
| Pesado para el servidor | Optimizado y cacheado |
| Sin control de quién pide qué | Identificado con API Key |

### En una frase

> Una API permite que **dos aplicaciones distintas se comuniquen** para
> **extraer** o **enviar** información, sin que ninguna necesite conocer
> cómo está construida la otra por dentro.

## 2. API vs Web Scraping — ¿cuándo usar cada una?

### El problema del Web Scraping

Cuando extraes datos de una web con Selenium o BeautifulSoup, estás
**simulando ser un humano navegando**. Eso significa:

- Descargas la **página completa** (HTML, CSS, imágenes, JavaScript) solo para
  sacar 3 datos.
- Envías **muchísimas solicitudes** que pueden **sobrecargar el servidor**,
  ralentizarlo o incluso "tumbar" la página.
- Si te pasas, **te bloquean la IP**.
- Si el equipo de diseño cambia un `<div>`, **tu script se rompe**.

### La solución: la API

Las empresas crean una API precisamente para evitar ese problema. Las solicitudes
van a un **servidor específico y optimizado**, no a la página web completa.

| | Web Scraping | API |
|---|---|---|
| **Qué recibes** | HTML completo | Solo los datos (JSON) |
| **Velocidad** | Lenta | Rápida |
| **Estabilidad** | Se rompe si cambia el diseño | Contrato estable y versionado |
| **Legalidad** | Zona gris | Explícitamente permitido |
| **Carga al servidor** | Alta | Baja |
| **Cuándo usarlo** | Cuando **no existe** API | **Siempre que exista** |

### Regla de oro

> **Antes de escribir una sola línea de scraping, verifica si la fuente ya tiene una API.**
> Te ahorra días de trabajo, y es el camino legal y eficiente.

### El caso que lo demuestra

Necesitas geolocalizar **50,000 colegios** del Perú.

- **Scraping de Google Maps:** inviable. Te bloquean la IP en el colegio número 200.
- **API de Geocoding:** miles de solicitudes estructuradas, automáticas y legales.

Ese salto de escala es exactamente la razón por la que existen las APIs.

## 3. Anatomía de una petición: URL base + endpoint + parámetros

Aquí está la parte que sorprende a todo el mundo:

> **Una API no es más que un link (una URL) al que le agregas argumentos.**

Nada mágico. Si pegas esa URL en tu navegador, ves los datos.

### Las 4 piezas

```
https://api.rawg.io/api/games?key=TU_KEY&ordering=-metacritic&page_size=5
└──────────┬─────────┘└──┬──┘ └──────────────────┬─────────────────────┘
     URL base        endpoint              parámetros (query string)
```

| Pieza | Qué es | Ejemplo |
|---|---|---|
| **URL base** | La dirección del servidor de la API | `https://api.rawg.io/api` |
| **Endpoint** | El "recurso" que quieres | `/games`, `/genres`, `/platforms` |
| **`?`** | Marca dónde empiezan los parámetros | `?` |
| **Parámetros** | Filtros y opciones, unidos con `&` | `key=...&page_size=5` |

### Los parámetros típicos

- **Autenticación:** `key`, `token`, `api_key` → quién eres
- **Filtro:** `search`, `genres`, `platforms`, `dates` → qué quieres
- **Orden:** `ordering=-metacritic` (el `-` significa descendente)
- **Tamaño:** `page_size=5` → cuántos resultados

### En Python no se escribe a mano

`requests` arma la URL por ti a partir de un diccionario. Es más legible y
maneja los caracteres especiales automáticamente:

```python
requests.get(BASE_URL + "/games", params={"key": API_KEY, "page_size": 5})
```

In [ ]:
# =============================================================
# CONFIGURACIÓN — la usaremos en todo el notebook
# =============================================================

import requests
import pandas as pd

# [Teoría §4] La API Key es tu contraseña de acceso.
# NUNCA se sube a GitHub: se mantiene como variable local o en un .env
API_KEY  = "TU_API_KEY_AQUI"

# [Teoría §3] La URL base: la dirección del servidor de la API
BASE_URL = "https://api.rawg.io/api"

In [ ]:
# -------------------------------------------------------------
# MINI-DEMO §3 — Ver cómo se arma la URL "a mano"
# -------------------------------------------------------------
# Esto es EXACTAMENTE lo que requests construye por debajo.
# Si copias el resultado y lo pegas en tu navegador, verás el JSON.

endpoint   = "/games"
parametros = {"key": API_KEY, "ordering": "-metacritic", "page_size": 5}

# Unimos cada par clave=valor con "&" — así se forma la query string
query = "&".join([f"{k}={v}" for k, v in parametros.items()])

print("URL base   :", BASE_URL)
print("Endpoint   :", endpoint)
print("Parámetros :", parametros)
print()
print("URL final  :", f"{BASE_URL}{endpoint}?{query}")

## 4. La API Key — tu contraseña de acceso

Una **API Key** es una cadena de texto que **identifica quién está haciendo la solicitud**.

> Es la **llave** que abre la puerta. Sin ella, la API no te reconoce y **no te
> devuelve nada** — aunque la API sea completamente gratuita.

### ¿Para qué sirve realmente?

| Función | Explicación |
|---|---|
| **Identificar** | La API sabe que eres tú y no otro |
| **Limitar** | Te aplica tu cuota (ej. 20,000 req/mes) |
| **Cobrar** | En APIs de pago, mide tu consumo |
| **Bloquear** | Si abusas, te desactivan la key (no toda la API) |

### Cómo viaja la key

Hay dos formas, según la API:

```python
# Forma A — como parámetro en la URL (RAWG, Finnhub, Google Maps)
requests.get(url, params={"key": API_KEY})

# Forma B — en los headers (GitHub, OpenAI, Anthropic)
requests.get(url, headers={"Authorization": f"token {TOKEN}"})
```

### Reglas de seguridad (esto se evalúa)

1. **Nunca** la escribas directamente en un notebook que vas a subir a GitHub.
2. Usa variables de entorno (`os.getenv`) o un archivo `.env` en el `.gitignore`.
3. Si se te filtró: **revócala y genera una nueva**. Es gratis.
4. En este notebook la dejamos como `"TU_API_KEY_AQUI"` justamente por eso.

## 5. Métodos HTTP: GET, POST, PUT, DELETE

Los métodos HTTP le indican a la API **qué acción quieres realizar** sobre los datos.
Son los **verbos** del idioma que hablan las APIs.

| Método | Acción | Ejemplo cotidiano | ¿Modifica datos? |
|---|---|---|---|
| **GET** | Obtener | Ver el catálogo de videojuegos | No |
| **POST** | Crear | Registrar un usuario nuevo | Sí |
| **PUT** | Actualizar | Editar tu perfil | Sí |
| **DELETE** | Eliminar | Borrar tu cuenta | Sí |

---

### GET — Obtener datos

Pide información al servidor. **No modifica nada.**
*Analogía:* abrir el catálogo de una tienda para ver los productos.

```python
respuesta = requests.get(
    "https://api.rawg.io/api/games",
    params={"key": API_KEY, "page_size": 5}
)
```

- Los parámetros van **en la URL**
- No envía *body*
- Se puede repetir mil veces sin efectos secundarios
- **Es el 95% de lo que harás en análisis de datos**

---

### POST — Crear datos

Envía datos nuevos para crear un recurso.
*Analogía:* llenar un formulario de registro por primera vez.

```python
nuevo_usuario = {"nombre": "Paul", "email": "paul@email.com"}

respuesta = requests.post(
    "https://api.ejemplo.com/usuarios",
    json=nuevo_usuario,                          # los datos van en el BODY
    headers={"Authorization": "token TU_TOKEN"}
)
# 201 = creado exitosamente
```

---

### PUT — Actualizar datos

Modifica un recurso **existente**, reemplazándolo por completo.
*Analogía:* editar y guardar tu perfil en una red social.

```python
respuesta = requests.put(
    "https://api.ejemplo.com/usuarios/42",   # el ID va en la URL
    json={"nombre": "Paul Melo", "ubicacion": "Lima, Perú"},
    headers={"Authorization": "token TU_TOKEN"}
)
# 200 = actualizado exitosamente
```

> **PATCH vs PUT:** `PATCH` actualiza solo los campos que le mandas;
> `PUT` reemplaza el objeto completo.

---

### DELETE — Eliminar datos

Borra un recurso de forma permanente.
*Analogía:* dar de baja tu cuenta.

```python
respuesta = requests.delete(
    "https://api.ejemplo.com/usuarios/42",
    headers={"Authorization": "token TU_TOKEN"}
)
# 204 = eliminado, sin contenido que devolver
```

---

> **En este curso usaremos casi exclusivamente `GET`**, porque las APIs públicas
> de datos (RAWG, Finnhub, Google Maps) son de **solo lectura**: nosotros
> consumimos su información, no editamos su base de datos.

In [ ]:
# -------------------------------------------------------------
# MINI-DEMO §5 — Nuestro primer GET real
# -------------------------------------------------------------
# Pedimos 3 juegos. Nada más. Solo para ver que la conexión funciona.

respuesta = requests.get(f"{BASE_URL}/games", params={
    "key": API_KEY,
    "page_size": 3
})

print("Método usado :", respuesta.request.method)   # GET
print("URL llamada  :", respuesta.url)              # la URL que armó requests
print("Código       :", respuesta.status_code)      # [Teoría §6]

## 6. Códigos de estado — ¿salió bien o mal?

Cada respuesta trae un `status_code`: un número de 3 dígitos que resume
**qué pasó con tu solicitud**. Se lee por su primer dígito:

| Rango | Significado | Traducción |
|---|---|---|
| **2xx** | Éxito | "Todo bien, aquí tienes" |
| **4xx** | Error **tuyo** | "Te equivocaste al pedir" |
| **5xx** | Error **del servidor** | "Me equivoqué yo, reintenta" |

### Los que vas a ver de verdad

| Código | Nombre | Qué hacer |
|---|---|---|
| **200** | OK | Todo bien, procesa el JSON |
| **201** | Created | Tu POST creó el recurso |
| **401** | Unauthorized | Falta la API Key o está mal escrita |
| **403** | Forbidden | Tienes key, pero no permiso para eso |
| **404** | Not Found | El endpoint o el ID no existe |
| **429** | Too Many Requests | Te pasaste del límite — espera |
| **500** | Internal Server Error | Falla del servidor, no tuya |

### Buena práctica: valida antes de procesar

```python
if respuesta.status_code == 200:
    data = respuesta.json()
else:
    print(f"Error {respuesta.status_code}: {respuesta.text}")
```

> Si haces `respuesta.json()` sobre un error 401, obtendrás un `KeyError`
> confuso más adelante. **Valida primero, procesa después.**

In [ ]:
# -------------------------------------------------------------
# MINI-DEMO §6 — Provocar un error a propósito
# -------------------------------------------------------------
# Mandamos una key inválida para ver cómo responde la API.
# Aprender a leer errores vale tanto como aprender a leer datos.

respuesta_mala = requests.get(f"{BASE_URL}/games", params={"key": "key_invalida_123"})

print("Código recibido:", respuesta_mala.status_code)

# El patrón que deberías usar SIEMPRE en tu código de producción
if respuesta_mala.status_code == 200:
    print("OK — datos listos para procesar")
else:
    print("Algo falló. Mensaje del servidor:")
    print(respuesta_mala.text[:200])

## 7. JSON — diccionarios y listas disfrazados

Cuando haces una solicitud a la API, la respuesta viene en formato **JSON**
(*JavaScript Object Notation*). Es el formato estándar en el que se empaqueta
la información que viaja por la web.

> **Buenas noticias:** un JSON **no es más que una colección de
> diccionarios y listas de Python.** Ya sabes manejarlo.

| En JSON | En Python |
|---|---|
| `{ }` object | `dict` |
| `[ ]` array | `list` |
| `"texto"` | `str` |
| `42` / `4.5` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

### El método `.json()`

```python
respuesta = requests.get(url, params=...)   # objeto Response
data      = respuesta.json()                # dict de Python listo para usar
```

### La estructura típica de RAWG

```python
{
  "count": 890000,             # total de juegos en la base de datos
  "next":  "...?page=2",       # link a la siguiente página
  "results": [                 # LISTA de juegos
      {
        "name":       "The Witcher 3",
        "rating":     4.66,
        "metacritic": 92,
        "genres":     [ {"name": "Action"}, {"name": "RPG"} ]   # lista dentro del dict
      },
      { ... }
  ]
}
```

### Cómo se navega

```python
data["count"]                          # un valor suelto
data["results"]                        # la lista completa
data["results"][0]                     # el primer juego
data["results"][0]["name"]             # su nombre
data["results"][0]["genres"][0]["name"]  # el nombre de su primer género
```

> **Truco para no perderte:** usa `data.keys()` para ver qué llaves tienes,
> y `type(...)` para saber si estás parado en un `dict` o en una `list`.

In [ ]:
# -------------------------------------------------------------
# MINI-DEMO §7 — Explorar el JSON paso a paso
# -------------------------------------------------------------

respuesta = requests.get(f"{BASE_URL}/games", params={"key": API_KEY, "page_size": 3})
data = respuesta.json()

# Paso 1: ¿qué tipo de objeto es? -> dict
print("Tipo de 'data'      :", type(data))

# Paso 2: ¿qué llaves tiene ese diccionario?
print("Llaves disponibles  :", list(data.keys()))

# Paso 3: 'count' es un valor suelto
print("Total de juegos     :", data["count"])

# Paso 4: 'results' es una LISTA de diccionarios
print("Tipo de 'results'   :", type(data["results"]))
print("Juegos en esta página:", len(data["results"]))

# Paso 5: entramos al primer juego y sacamos un campo
primer_juego = data["results"][0]
print("Primer juego        :", primer_juego["name"])

# Paso 6: lista dentro de un dict dentro de una lista
print("Su primer género    :", primer_juego["genres"][0]["name"])

## 8. De JSON a DataFrame

Un JSON está bien para la máquina, pero **para analizar datos queremos una tabla**.
El puente entre ambos mundos es esta línea:

```python
df = pd.DataFrame([ {campos que quiero} for item in data["results"] ])
```

### Por qué un *list comprehension* y no `pd.DataFrame(data["results"])`

Si le pasas los resultados crudos a pandas, obtienes **30 columnas**, varias de
ellas con listas y diccionarios dentro de las celdas — imposible de analizar.

Con el list comprehension tú decides:

- **Qué columnas** quieres (`name`, `rating`, `metacritic`)
- **Cómo aplanar** los campos anidados

### Aplanar campos anidados

`genres` es una lista de diccionarios. Para volverla texto:

```python
", ".join([g["name"] for g in juego["genres"]])   # -> "Action, RPG"
```

### El patrón completo, de principio a fin

```python
respuesta = requests.get(url, params={...})     # 1. GET          [§5]
data      = respuesta.json()                    # 2. JSON -> dict [§7]
df        = pd.DataFrame([...])                 # 3. dict -> tabla [§8]
df.to_csv("salida.csv", index=False)            # 4. exportar
```

**Estos 4 pasos son el 90% del trabajo con APIs.** Todo el caso práctico
de la Parte 2 es este patrón repetido con distintos filtros.

In [ ]:
# -------------------------------------------------------------
# MINI-DEMO §8 — El mismo JSON, ahora como tabla
# -------------------------------------------------------------

respuesta = requests.get(f"{BASE_URL}/games", params={"key": API_KEY, "page_size": 5})
data = respuesta.json()

# Elegimos SOLO los campos que nos interesan y aplanamos 'genres'
df_demo = pd.DataFrame([{
    "name":       g["name"],
    "rating":     g["rating"],
    "metacritic": g["metacritic"],
    "genres":     ", ".join([genre["name"] for genre in g["genres"]])
} for g in data["results"]])

df_demo

## 9. Cómo obtener tu API Key de RAWG (paso a paso)

**RAWG** es la base de datos de videojuegos más grande de internet: más de
**800,000 juegos** con ratings, géneros, plataformas, tiendas y fechas de lanzamiento.

### Por qué la usamos en clase

| Motivo | Detalle |
|---|---|
| **Gratis** | 20,000 requests al mes, sin tarjeta |
| **Registro simple** | Solo correo, la key llega al instante |
| **Datos ricos** | Ratings, metacritic, géneros, plataformas, tiendas, fechas |
| **JSON limpio** | Estructura predecible y bien documentada |
| **Motivante** | Todos conocen los videojuegos → el análisis se entiende solo |

### Pasos

**Paso 1 — Crear la cuenta**
1. Entra a **https://rawg.io/apidocs**
2. Haz clic en **Get API Key**
3. Regístrate con tu correo (o con Google / GitHub)

**Paso 2 — Copiar la key**
Apenas confirmas el registro, la key aparece en pantalla. Se ve así:

```
a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6
```

**Paso 3 — Pegarla en el notebook**
```python
API_KEY = "a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6"
```

**Paso 4 — Probar que funciona**
Pega esto en tu navegador reemplazando tu key. Si ves un JSON, ya estás listo:
```
https://api.rawg.io/api/games?key=TU_KEY&page_size=1
```

### Límites del plan gratuito

| Situación | Requests |
|---|---|
| Sin API Key | 0 — la API te rechaza |
| Con API Key (free) | 20,000 al mes |
| `page_size` máximo | 40 resultados por request |

> Con 20,000 requests al mes te sobra: toda la tarea de la Parte 2
> consume **menos de 20 requests**.

### Parámetros más usados del endpoint `/games`

| Parámetro | Qué hace | Ejemplo |
|---|---|---|
| `search` | Busca por nombre | `search=Zelda` |
| `ordering` | Ordena (`-` = descendente) | `ordering=-metacritic` |
| `page_size` | Cuántos resultados | `page_size=10` |
| `platforms` | Filtra por plataforma (ID) | `platforms=4` (PC) |
| `stores` | Filtra por tienda (ID) | `stores=1` (Steam) |
| `genres` | Filtra por género (slug) | `genres=action` |
| `dates` | Rango de fechas | `dates=2020-01-01,2020-12-31` |

## 10. Escalabilidad, límites y buenas prácticas

### El problema real

Una API te da **cuota limitada**. RAWG: 20,000 requests/mes.
Si dentro de un `for` haces una llamada por cada elemento de una lista de 5,000
juegos, quemaste el 25% de tu mes en un solo `for` mal escrito.

> **Lo que no se mide, no se controla.** Por eso lo primero que haremos en el
> caso práctico es **contar nuestros propios requests**.

### Buenas prácticas

| Práctica | Por qué |
|---|---|
| **Contar tus requests** | Saber cuánta cuota consumes |
| **Pedir en lote** | Un `page_size=20` cuesta 1 request; 20 llamadas cuestan 20 |
| **Guardar la respuesta** | No vuelvas a pedir lo mismo — usa la variable o un CSV |
| **Validar `status_code`** | Detectar el 429 antes de que rompa todo |
| **Nunca subir la key** | Seguridad y cuota |

### Por qué envolvemos `requests` en una clase

En el caso práctico no llamaremos `requests.get()` directamente, sino
`client.get()`. La clase es un **envoltorio (wrapper)** que hace lo mismo que
`requests`, pero además **suma 1 a un contador** en cada llamada.

```python
class RAWGClient:
    def __init__(self):
        self.request_count = 0          # contador que arranca en 0

    def get(self, url, params=None):
        self.request_count += 1         # +1 en CADA llamada
        return requests.get(url, params=params)
```

Ventaja: al final del análisis puedes responder con certeza
*"usé exactamente 12 requests"*. Ese es el mismo principio con el que se
audita el consumo de APIs de pago (OpenAI, Google Maps) en producción.

---
# PARTE 2 — CASO PRÁCTICO
# Task 2: API REST — RAWG Video Games Database
---

A partir de aquí aplicamos **todo lo anterior** sobre un caso real.

### Mapa: teoría → práctica

| Concepto de la Parte 1 | Dónde lo verás aplicado |
|---|---|
| §3 URL base + endpoint + params | Cada llamada `client.get(f"{BASE_URL}/games", params={...})` |
| §4 API Key | El parámetro `"key": API_KEY` en todas las llamadas |
| §5 Método GET | Todas las consultas — solo leemos, nunca escribimos |
| §6 Códigos de estado | Validación antes de procesar en A1 |
| §7 JSON = dicts + listas | `data["count"]`, `data["results"]`, `g["genres"][0]["name"]` |
| §8 JSON → DataFrame | Todos los `pd.DataFrame([... for g in ...])` |
| §10 Control de cuota | `RAWGClient` y el reporte final de requests |

### Estructura de la tarea

- **Part A — General Exploration:** una consulta simple para entender la base
- **Part B — Category Analysis:** filtros y ordenamiento
- **Part C — Comparisons:** múltiples llamadas, agregaciones y exportación
- **Part D — Insights & Conclusions:** interpretación de los resultados

In [ ]:
# =============================================================
# SETUP DEL CASO PRÁCTICO
# =============================================================
import requests
import pandas as pd

# [Teoría §4] API Key personal de RAWG — identifica quién hace las solicitudes.
# IMPORTANTE: no subir esta key a GitHub (mantenerla como variable local)
API_KEY  = "TU_API_KEY_AQUI"

# [Teoría §3] URL base de la API — a esto le agregaremos endpoints y parámetros
BASE_URL = "https://api.rawg.io/api"

In [ ]:
# [Teoría §10] Envolvemos la librería 'requests' en una clase propia.
# Su objetivo es llevar un conteo automático de cuántas solicitudes (requests)
# hacemos a la API durante toda la sesión, para controlar nuestra cuota mensual.

class RAWGClient:
    def __init__(self):
        self.request_count = 0            # contador de solicitudes, arranca en 0

    def get(self, url, params=None):
        self.request_count += 1           # cada llamada suma 1 al contador
        return requests.get(url, params=params)   # [Teoría §5] método GET

    def resumen_requests(self):
        return self.request_count

In [ ]:
# Instanciamos el cliente — a partir de aquí usamos 'client.get()'
# en lugar de 'requests.get()' para que el conteo funcione automáticamente
client = RAWGClient()

## Part A — General Exploration

### A1 — How many games does RAWG have registered in total?

In [ ]:
# [Teoría §5] Hacemos un GET al endpoint /games sin filtros
# [Teoría §4] El único parámetro obligatorio es la API Key
response = client.get(f"{BASE_URL}/games", params={"key": API_KEY})

# [Teoría §6] Validamos el código de estado ANTES de procesar la respuesta
print("Status code:", response.status_code)

# [Teoría §7] .json() convierte la respuesta en un diccionario de Python
data = response.json()

# [Teoría §7] 'count' es un valor suelto del diccionario raíz:
# contiene el total de juegos en toda la base de datos, no solo los de esta página
total_games = data["count"]

print(f"Total games registered in RAWG: {total_games:,}")

## Part B — Category Analysis

### B1 — Top 5 highest rated games of all time according to Metacritic

In [ ]:
# [Teoría §3] Agregamos parámetros de filtro y orden a la misma URL base:
#   ordering="-metacritic" -> el guion indica orden DESCENDENTE (mayor a menor)
#   page_size=5            -> limita la respuesta a los 5 primeros resultados
response = client.get(f"{BASE_URL}/games", params={
    "key":       API_KEY,
    "ordering":  "-metacritic",
    "page_size": 5
})
data = response.json()

# [Teoría §8] Convertimos el JSON en tabla con un list comprehension,
# eligiendo solo las 3 columnas que nos interesan para el análisis
df_b1 = pd.DataFrame([{
    "name":       g["name"],
    "rating":     g["rating"],
    "metacritic": g["metacritic"]
} for g in data["results"]])          # 'results' es la LISTA de juegos

df_b1

### B2 — Top 10 best games available on Steam (store_id=1)

In [ ]:
# [Teoría §3] Mismo endpoint, un filtro adicional:
#   stores=1 -> ID de Steam dentro del catálogo de tiendas de RAWG
# Los IDs se consultan en el endpoint /stores
response = client.get(f"{BASE_URL}/games", params={
    "key":       API_KEY,
    "stores":    1,
    "ordering":  "-metacritic",
    "page_size": 10
})
data = response.json()

# [Teoría §8] Reutilizamos exactamente el mismo patrón JSON -> DataFrame.
# Cambian los filtros, NO cambia la forma de procesar la respuesta.
df_b2 = pd.DataFrame([{
    "name":       g["name"],
    "rating":     g["rating"],
    "metacritic": g["metacritic"]
} for g in data["results"]])

df_b2

## Part C — Comparisons

### C1 — Top 5 games on PC vs Top 5 on PS5

In [ ]:
# Comparamos dos plataformas. El ordenamiento se hace por "metacritic"
# (nota de crítica profesional); también se podría usar "rating" (nota de usuarios).

# ---------- Top 5 PC (platform_id=4) ----------
# [Teoría §10] Una llamada por plataforma: 2 requests en total, no 10
response_pc = client.get(f"{BASE_URL}/games", params={
    "key":       API_KEY,
    "platforms": 4,
    "ordering":  "-metacritic",
    "page_size": 5
})

df_pc = pd.DataFrame([{
    "name":       g["name"],
    "rating":     g["rating"],
    "metacritic": g["metacritic"]
} for g in response_pc.json()["results"]])

# Insertamos la columna 'platform' en la posición 0 (primera columna) para poder
# identificar a qué plataforma pertenece cada fila cuando unamos ambas tablas
df_pc.insert(0, "platform", "PC")


# ---------- Top 5 PS5 (platform_id=187) ----------
response_ps5 = client.get(f"{BASE_URL}/games", params={
    "key":       API_KEY,
    "platforms": 187,
    "ordering":  "-metacritic",
    "page_size": 5
})

df_ps5 = pd.DataFrame([{
    "name":       g["name"],
    "rating":     g["rating"],
    "metacritic": g["metacritic"]
} for g in response_ps5.json()["results"]])

df_ps5.insert(0, "platform", "PS5")


# ---------- Comparación ----------
# concat apila ambos DataFrames uno debajo del otro
df_c1 = pd.concat([df_pc, df_ps5], ignore_index=True)

# Promedio de metacritic de cada plataforma para poder compararlas
print(f"PC average metacritic:  {df_pc['metacritic'].mean():.2f}")
print(f"PS5 average metacritic: {df_ps5['metacritic'].mean():.2f}")

# Determinamos cuál plataforma tiene el mayor promedio
winner = "PC" if df_pc['metacritic'].mean() > df_ps5['metacritic'].mean() else "PS5"
print(f"\nPlatform with highest rated games: {winner}")

df_c1

### C2 — Comparison table of 3 famous games

In [ ]:
# Lista de 3 juegos famosos que queremos comparar
famous_games = ["Red Dead Redemption 2", "The Witcher 3: Wild Hunt", "Grand Theft Auto V"]

games_data = []
for game_name in famous_games:
    # [Teoría §3] El parámetro 'search' busca por nombre.
    # page_size=1 -> solo el resultado más relevante
    # [Teoría §10] Este for hace 1 request por juego: 3 en total. Con 5,000 juegos
    # este mismo patrón consumiría 5,000 requests — hay que tenerlo presente.
    response = client.get(f"{BASE_URL}/games", params={
        "key":       API_KEY,
        "search":    game_name,
        "page_size": 1
    })

    # Tomamos el primer (y único) resultado de la búsqueda
    g = response.json()["results"][0]

    games_data.append({
        "name":       g["name"],
        "rating":     g["rating"],
        "metacritic": g["metacritic"],
        # [Teoría §8] 'genres' y 'platforms' son LISTAS DE DICCIONARIOS.
        # Las aplanamos a texto con join + list comprehension.
        "genres":    ", ".join([genre["name"] for genre in g["genres"]]),
        "platforms": ", ".join([p["platform"]["name"] for p in g["platforms"]])
    })

df_c2 = pd.DataFrame(games_data)

# Nota: la columna "platforms" tiene muchos valores porque son juegos muy famosos
# lanzados en casi todas las consolas.
df_c2

### C3 — Average rating by genre (at least 4 different genres)

In [ ]:
# Aquí el ordenamiento se hace por "rating" (nota de USUARIOS), no por metacritic,
# porque la consigna pide el promedio "according to users".

# Los géneros se identifican por su 'slug' (nombre en formato URL), no por ID
genres = ["action", "role-playing-games-rpg", "shooter", "strategy"]

genre_data = []
for genre in genres:
    # [Teoría §3] Un request por género: pedimos su top 5 por rating
    response = client.get(f"{BASE_URL}/games", params={
        "key":       API_KEY,
        "genres":    genre,
        "ordering":  "-rating",
        "page_size": 5
    })
    results = response.json()["results"]

    # Defensa: si la API no devolvió resultados para este género, lo saltamos.
    # Evita un ZeroDivisionError al calcular el promedio.
    if len(results) == 0:
        continue

    # Promedio de rating de los top 5 juegos del género
    avg_rating = sum([g["rating"] for g in results]) / len(results)

    genre_data.append({
        "genre":          genre,
        "average_rating": round(avg_rating, 2)
    })

df_c3 = pd.DataFrame(genre_data)

# Ordenamos de mayor a menor promedio y reiniciamos el índice
df_c3 = df_c3.sort_values("average_rating", ascending=False).reset_index(drop=True)

# iloc[0] accede a la primera fila -> el género con mayor promedio
best_genre = df_c3.iloc[0]["genre"]
print(f"Genre with highest average rating: {best_genre}")

df_c3

### C4 — Best games from 3 different years

In [ ]:
# Años a comparar — se puede elegir cualquiera
years = [2020, 2021, 2022]

year_data = []
for year in years:
    # [Teoría §3] El parámetro 'dates' filtra por rango de fechas.
    # Formato: 'YYYY-MM-DD,YYYY-MM-DD' (desde el inicio hasta el fin del año)
    response = client.get(f"{BASE_URL}/games", params={
        "key":       API_KEY,
        "dates":     f"{year}-01-01,{year}-12-31",
        "ordering":  "-metacritic",
        "page_size": 5     # top 5 de cada año para el cálculo del promedio
    })
    results = response.json()["results"]

    # Ojo: algunos juegos tienen metacritic = None (null en el JSON).
    # Por eso filtramos con 'if g["metacritic"]' tanto en la suma como en el conteo.
    avg_metacritic = (
        sum([g["metacritic"] for g in results if g["metacritic"]])
        / len([g for g in results if g["metacritic"]])
    )

    year_data.append({
        "year":               year,
        "average_metacritic": round(avg_metacritic, 2)
    })

df_c4 = pd.DataFrame(year_data)

# Ordenamos de mayor a menor promedio de metacritic
df_c4 = df_c4.sort_values("average_metacritic", ascending=False).reset_index(drop=True)

best_year = df_c4.iloc[0]["year"]
print(f"Year with highest average metacritic score: {best_year}")

df_c4

### C5 — Export top 20 games of all time to CSV

In [ ]:
import os

# [Teoría §10] Un solo request con page_size=20 en lugar de 20 requests sueltos.
# El ordenamiento se hace por "metacritic"; también podría usarse "rating".
response = client.get(f"{BASE_URL}/games", params={
    "key":       API_KEY,
    "ordering":  "-metacritic",
    "page_size": 20
})
results = response.json()["results"]

# [Teoría §8] JSON -> DataFrame con las columnas que pide la consigna.
# En 'main_genre' usamos un condicional por si un juego no tiene géneros
# registrados (la lista vendría vacía y g["genres"][0] lanzaría IndexError).
df_c5 = pd.DataFrame([{
    "name":         g["name"],
    "rating":       g["rating"],
    "metacritic":   g["metacritic"],
    "release_date": g["released"],
    "main_genre":   g["genres"][0]["name"] if g["genres"] else ""
} for g in results])

# Creamos la carpeta 'output/' si no existe (exist_ok=True evita error si ya existe)
os.makedirs("output", exist_ok=True)

# [Teoría §10] Guardamos el resultado: si mañana necesitamos estos datos,
# leemos el CSV en vez de gastar otro request
df_c5.to_csv("output/top20_rawg.csv", index=False)

print("CSV saved in output/top20_rawg.csv")

# Mostramos solo las primeras 5 filas como verificación
df_c5.head(5)

## Part D — Insights & Conclusions

### D1 — Personal Conclusions

**What was the most interesting thing you found in the data?**
The most interesting finding was that PC has a higher average Metacritic score than
PS5, suggesting that PC tends to have more critically acclaimed titles overall.

**Which genre or platform surprised you the most and why?**
The RPG genre surprised me the most because despite being a niche genre compared to
action or shooter, it produced the highest average user rating among all genres analyzed.

**What other question would you ask this API if you had more time?**
I would analyze the relationship between a game's Metacritic score and its commercial
success (number of reviews), to determine if critically acclaimed games are also the
most popular ones.

In [ ]:
# [Teoría §10] Reporte final de consumo: aquí se paga el diseño del wrapper.
# Sin la clase RAWGClient tendríamos que contar las llamadas a mano.
total_requests = client.resumen_requests()

print(f"How many requests did you use in total? -> Total API requests used: {total_requests}")
print(f"Cuota mensual del plan free: 20,000 -> consumido: {total_requests/20000:.3%}")

---

## Cierre — Lo que hay que llevarse de esta clase

1. **Una API es un intermediario**, no una base de datos abierta. Pides por la
   puerta correcta y te dan solo lo que corresponde.
2. **Si existe una API, no hagas scraping.** Es más rápido, más estable y legal.
3. **Una petición es una URL con argumentos.** Nada más.
4. **La API Key es tu identidad.** Sin ella no entras; si la filtras, la revocas.
5. **La respuesta es JSON = diccionarios y listas** que ya sabes manejar.
6. **El patrón de trabajo son 4 pasos:** `GET` → `.json()` → `DataFrame` → análisis.
7. **La cuota es finita.** Cuenta tus requests y guarda lo que ya pediste.

> Todo el caso práctico de la Parte 2 —8 consultas distintas, comparaciones entre
> plataformas, géneros y años— se resolvió repitiendo esos mismos 4 pasos
> con distintos filtros.